# Concept Extraction Analysis

Analyze `concept-extraction` outputs: validation site/alpha selection, held-out steering effects, and random-direction controls.

## Runtime Setup

This notebook is now visualization-only. The heavyweight activation-geometry and BLiMP transfer computations should be precomputed with `animacy-circuit/scripts/executable/run_concept_analysis_artifacts.py`, which saves CSV/JSON artifacts into the selected concept-extraction run directory.

A GPU kernel is no longer required just to inspect the results. The notebook only needs the saved artifacts from that script run.


In [1]:
from __future__ import annotations
import os

import json
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import Markdown, display

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 160)

In [2]:
def find_project_root(start: Path | None = None) -> Path:
    anchor = (start or Path.cwd()).resolve()
    for base in (anchor, *anchor.parents):
        candidate = base if base.name == "animacy-circuit" else base / "animacy-circuit"
        if (candidate / "results").is_dir() and (candidate / "scripts").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate animacy-circuit project root.")


PROJECT_ROOT = find_project_root()
RESULTS_ROOT = PROJECT_ROOT / "results" / "concept_extraction"
PROJECT_ROOT

PosixPath('/gpfs/home4/spunzo/grammatical-circuits/animacy-circuit')

In [3]:
CONDA_ENV_PREFIX = Path("/home/spunzo/.conda/envs/grammatical-circuits")
CONDA_LIB = CONDA_ENV_PREFIX / "lib"
ld_library_entries = [entry for entry in os.environ.get("LD_LIBRARY_PATH", "").split(":") if entry]
conda_lib_first = bool(ld_library_entries) and Path(ld_library_entries[0]).resolve() == CONDA_LIB.resolve()
runtime_status = {
    "python_executable": os.environ.get("PYTHON_EXECUTABLE") or os.sys.executable,
    "ld_library_path_head": ld_library_entries[:3],
    "conda_lib_first": conda_lib_first,
}
display(Markdown(
    "Runtime check: conda `lib` first on `LD_LIBRARY_PATH` = "
    + ("`True`" if conda_lib_first else "`False`")
))
if not conda_lib_first:
    display(Markdown(
        "Launch this notebook with "
        "`animacy-circuit/scripts/notebooks/launch_concept_analysis_cpu.sh` "
        "or start the kernel with `LD_LIBRARY_PATH=/home/spunzo/.conda/envs/grammatical-circuits/lib:$LD_LIBRARY_PATH`."
    ))
runtime_status


Runtime check: conda `lib` first on `LD_LIBRARY_PATH` = `False`

Launch this notebook with `animacy-circuit/scripts/notebooks/launch_concept_analysis_cpu.sh` or start the kernel with `LD_LIBRARY_PATH=/home/spunzo/.conda/envs/grammatical-circuits/lib:$LD_LIBRARY_PATH`.

{'python_executable': '/home/spunzo/.conda/envs/grammatical-circuits/bin/python',
 'ld_library_path_head': [],
 'conda_lib_first': False}

## Select Run

Set `MODEL_SLUG` and optionally `RUN_DAY`. Leave `RUN_DAY = None` to pick the latest run directory for the model.

In [4]:
MODEL_SLUG = "gpt2"
RUN_DAY = None  # Example: "2026-05-30" or "smoke_test"


def latest_child_dir(path: Path) -> Path:
    dirs = [child for child in path.iterdir() if child.is_dir()]
    if not dirs:
        raise FileNotFoundError(f"No run directories found under {path}")
    return max(dirs, key=lambda p: p.stat().st_mtime)


model_root = RESULTS_ROOT / MODEL_SLUG
RUN_DIR = model_root / RUN_DAY if RUN_DAY is not None else latest_child_dir(model_root)
display(Markdown(f"**Run directory:** `{RUN_DIR}`"))
sorted(path.name for path in RUN_DIR.glob("*"))

**Run directory:** `/gpfs/home4/spunzo/grammatical-circuits/animacy-circuit/results/concept_extraction/gpt2/2026-05-30`

['analysis_pca_points_2026-06-04_152924.csv',
 'analysis_pca_points_2026-06-04_161430.csv',
 'analysis_projection_rows_2026-06-04_152924.csv',
 'analysis_projection_rows_2026-06-04_161430.csv',
 'analysis_projection_summary_2026-06-04_152924.json',
 'analysis_projection_summary_2026-06-04_161430.json',
 'blimp_transfer_alignment_failures_2026-06-04_152924.csv',
 'blimp_transfer_alignment_failures_2026-06-04_161430.csv',
 'blimp_transfer_alignment_failures_2026-06-04_164231.csv',
 'blimp_transfer_alignment_failures_2026-06-04_165757.csv',
 'blimp_transfer_alignment_failures_2026-06-06_192137.csv',
 'blimp_transfer_alignment_failures_2026-06-06_202932.csv',
 'blimp_transfer_alignment_failures_2026-06-06_203315.csv',
 'blimp_transfer_random_summary_2026-06-04_152924.csv',
 'blimp_transfer_random_summary_2026-06-04_161430.csv',
 'blimp_transfer_random_summary_2026-06-04_164231.csv',
 'blimp_transfer_random_summary_2026-06-04_165757.csv',
 'blimp_transfer_random_summary_2026-06-06_192137.cs

In [5]:
def latest_file(pattern: str, required: bool = False) -> Path | None:
    matches = sorted(RUN_DIR.glob(pattern), key=lambda p: p.stat().st_mtime)
    if not matches:
        if required:
            raise FileNotFoundError(f"No file matching {pattern!r} in {RUN_DIR}")
        return None
    return matches[-1]


def read_csv_optional(path: Path | None, label: str) -> pd.DataFrame:
    if path is None:
        display(Markdown(f"Missing optional file: `{label}`"))
        return pd.DataFrame()
    return pd.read_csv(path)


def read_json_optional(path: Path | None, label: str) -> dict:
    if path is None:
        display(Markdown(f"Missing optional file: `{label}`"))
        return {}
    return json.loads(path.read_text())


analysis_manifest_path = latest_file("concept_analysis_artifacts_*.json")
analysis_manifest_preview = (
    json.loads(analysis_manifest_path.read_text()) if analysis_manifest_path is not None else {}
)
manifest_paths = {
    key: Path(value)
    for key, value in (analysis_manifest_preview.get("paths") or {}).items()
    if value
}


def manifest_or_latest(name: str, pattern: str) -> Path | None:
    manifest_path = manifest_paths.get(name)
    if manifest_path is not None and manifest_path.exists():
        return manifest_path
    return latest_file(pattern)


paths = {
    "summary": latest_file("concept_extraction_summary_*.json"),
    "selected": latest_file("selected_site_*.json"),
    "site_summary": latest_file("concept_site_summary_*.csv"),
    "validation_sweep": latest_file("validation_sweep_*.csv"),
    "test_rows": latest_file("test_steering_rows_*.csv"),
    "random_control": latest_file("random_control_test_summary_*.csv"),
    "analysis_manifest": analysis_manifest_path,
    "analysis_projection_rows": manifest_or_latest("analysis_projection_rows", "analysis_projection_rows_*.csv"),
    "analysis_projection_summary": manifest_or_latest("analysis_projection_summary", "analysis_projection_summary_*.json"),
    "analysis_pca_points": manifest_or_latest("analysis_pca_points", "analysis_pca_points_*.csv"),
    "blimp_transfer_status": manifest_or_latest("blimp_transfer_status", "blimp_transfer_status_*.json"),
    "blimp_transfer_alignment_failures": manifest_or_latest("blimp_transfer_alignment_failures", "blimp_transfer_alignment_failures_*.csv"),
    "blimp_transfer_rows": manifest_or_latest("blimp_transfer_rows", "blimp_transfer_rows_*.csv"),
    "blimp_transfer_summary": manifest_or_latest("blimp_transfer_summary", "blimp_transfer_summary_*.csv"),
    "blimp_transfer_random_summary": manifest_or_latest("blimp_transfer_random_summary", "blimp_transfer_random_summary_*.csv"),
}
pd.DataFrame([{"artifact": key, "path": str(value) if value else None} for key, value in paths.items()])


,artifact,path
0,summary,/gpfs/home4/spunzo/grammatical-circuits/animacy-circuit/results/concept_extraction/gpt2/2026-05-30/concept_extraction_summary_2026-05-30_233603.json
1,selected,/gpfs/home4/spunzo/grammatical-circuits/animacy-circuit/results/concept_extraction/gpt2/2026-05-30/selected_site_2026-05-30_233603.json
2,site_summary,/gpfs/home4/spunzo/grammatical-circuits/animacy-circuit/results/concept_extraction/gpt2/2026-05-30/concept_site_summary_2026-05-30_233603.csv
3,validation_sweep,/gpfs/home4/spunzo/grammatical-circuits/animacy-circuit/results/concept_extraction/gpt2/2026-05-30/validation_sweep_2026-05-30_233603.csv
4,test_rows,/gpfs/home4/spunzo/grammatical-circuits/animacy-circuit/results/concept_extraction/gpt2/2026-05-30/test_steering_rows_2026-05-30_233603.csv
5,random_control,/gpfs/home4/spunzo/grammatical-circuits/animacy-circuit/results/concept_extraction/gpt2/2026-05-30/random_control_test_summary_2026-05-30_233603.csv
6,analysis_manifest,/gpfs/home4/spunzo/grammatical-circuits/animacy-circuit/results/concept_extraction/gpt2/2026-05-30/concept_analysis_artifacts_2026-06-06_203315.json
7,analysis_projection_rows,/gpfs/home4/spunzo/grammatical-circuits/animacy-circuit/results/concept_extraction/gpt2/2026-05-30/analysis_projection_rows_2026-06-04_161430.csv
8,analysis_projection_summary,/gpfs/home4/spunzo/grammatical-circuits/animacy-circuit/results/concept_extraction/gpt2/2026-05-30/analysis_projection_summary_2026-06-04_161430.json
9,analysis_pca_points,/gpfs/home4/spunzo/grammatical-circuits/animacy-circuit/results/concept_extraction/gpt2/2026-05-30/analysis_pca_points_2026-06-04_161430.csv


In [6]:
summary = read_json_optional(paths.get("summary"), "concept_extraction_summary_*.json")
selected_payload = read_json_optional(paths.get("selected"), "selected_site_*.json")
site_summary = read_csv_optional(paths.get("site_summary"), "concept_site_summary_*.csv")
validation = read_csv_optional(paths.get("validation_sweep"), "validation_sweep_*.csv")
test_rows = read_csv_optional(paths.get("test_rows"), "test_steering_rows_*.csv")
random_control = read_csv_optional(paths.get("random_control"), "random_control_test_summary_*.csv")
train_split = read_csv_optional(latest_file("train_split_*.csv"), "train_split_*.csv")
validation_split = read_csv_optional(latest_file("validation_split_*.csv"), "validation_split_*.csv")
test_split = read_csv_optional(latest_file("test_split_*.csv"), "test_split_*.csv")
analysis_manifest = read_json_optional(paths.get("analysis_manifest"), "concept_analysis_artifacts_*.json")
projection_df = read_csv_optional(paths.get("analysis_projection_rows"), "analysis_projection_rows_*.csv")
projection_summary_payload = read_json_optional(paths.get("analysis_projection_summary"), "analysis_projection_summary_*.json")
pca_points = read_csv_optional(paths.get("analysis_pca_points"), "analysis_pca_points_*.csv")
blimp_status = read_json_optional(paths.get("blimp_transfer_status"), "blimp_transfer_status_*.json")
blimp_alignment_failures = read_csv_optional(paths.get("blimp_transfer_alignment_failures"), "blimp_transfer_alignment_failures_*.csv")
blimp_results = read_csv_optional(paths.get("blimp_transfer_rows"), "blimp_transfer_rows_*.csv")
blimp_summary = read_csv_optional(paths.get("blimp_transfer_summary"), "blimp_transfer_summary_*.csv")
blimp_random_summary = read_csv_optional(paths.get("blimp_transfer_random_summary"), "blimp_transfer_random_summary_*.csv")


## Run Overview

In [7]:
overview = {
    "model": summary.get("model_name", MODEL_SLUG),
    "dataset_filter_model": summary.get("dataset_filter_model_name"),
    "split_counts": summary.get("split_counts"),
    "selected_hook": (summary.get("selected") or selected_payload.get("selected") or {}).get("hook_name"),
    "selected_alpha": (summary.get("selected") or selected_payload.get("selected") or {}).get("alpha"),
    "validation_effect": (summary.get("selected") or selected_payload.get("selected") or {}).get("signed_effect_mean"),
    "test_effect": summary.get("test_summary", {}).get("signed_effect_mean"),
    "random_control_mean": summary.get("random_control_summary", {}).get("signed_effect_mean"),
}
pd.DataFrame([overview]).T.rename(columns={0: "value"})

,value
model,gpt2
dataset_filter_model,gpt2
split_counts,"{'train': 2833, 'validation': 944, 'test': 944}"
selected_hook,blocks.0.hook_resid_pre
selected_alpha,5.0
validation_effect,3.078728
test_effect,3.062685
random_control_mean,0.059649


In [8]:
def uid_set(df: pd.DataFrame) -> set[str]:
    if df.empty or "uid" not in df:
        return set()
    return set(df["uid"].dropna().astype(str))


split_checks = pd.DataFrame(
    [
        {"check": "train_validation_uid_overlap", "count": len(uid_set(train_split) & uid_set(validation_split))},
        {"check": "train_test_uid_overlap", "count": len(uid_set(train_split) & uid_set(test_split))},
        {"check": "validation_test_uid_overlap", "count": len(uid_set(validation_split) & uid_set(test_split))},
    ]
)
split_checks

,check,count
0,train_validation_uid_overlap,0
1,train_test_uid_overlap,0
2,validation_test_uid_overlap,0


## Validation Sweep

`signed_effect_mean` is the selection objective: clean metric decrease plus corrupt metric increase, averaged per pair.

In [9]:
if validation.empty:
    display(Markdown("Validation sweep is not available yet. The run may still be in progress."))
else:
    display(validation.head(10))

,layer,hook_point,hook_name,site_key,train_examples,clean_mean_norm,corrupt_mean_norm,concept_norm,alpha,steering_vector_norm,normalize_concept_vector,example_count,clean_before_mean,clean_after_mean,corrupt_before_mean,corrupt_after_mean,clean_signed_effect_mean,corrupt_signed_effect_mean,signed_effect_mean,signed_effect_std,clean_flip_rate,corrupt_flip_rate,abs_alpha,selection_threshold,selection_eligible,selected
0,0,hook_resid_pre,blocks.0.hook_resid_pre,blocks__0__hook_resid_pre,2833,4.719832,4.851473,1.466795,5.0,1.0,True,944,1.516403,-1.715990,-1.054814,1.870249,3.232393,2.925062,3.078728,0.562842,0.980932,1.000000,5.0,2.770855,True,True
1,0,hook_resid_pre,blocks.0.hook_resid_pre,blocks__0__hook_resid_pre,2833,4.719832,4.851473,1.466795,7.5,1.0,True,944,1.516403,-1.181988,-1.054814,1.943992,2.698391,2.998806,2.848599,0.552895,0.961864,1.000000,7.5,2.770855,True,False
2,0,hook_resid_pre,blocks.0.hook_resid_pre,blocks__0__hook_resid_pre,2833,4.719832,4.851473,1.466795,10.0,1.0,True,944,1.516403,-0.885502,-1.054814,1.853395,2.401906,2.908209,2.655057,0.621482,0.876059,1.000000,10.0,2.770855,False,False
3,0,hook_resid_pre,blocks.0.hook_resid_pre,blocks__0__hook_resid_pre,2833,4.719832,4.851473,1.466795,3.0,1.0,True,944,1.516403,-1.095362,-1.054814,1.505747,2.611766,2.560561,2.586163,0.579525,0.858051,1.000000,3.0,2.770855,False,False
4,0,hook_resid_pre,blocks.0.hook_resid_pre,blocks__0__hook_resid_pre,2833,4.719832,4.851473,1.466795,2.0,1.0,True,944,1.516403,0.019327,-1.054814,0.970722,1.497076,2.025536,1.761306,0.579081,0.566737,0.900424,2.0,2.770855,False,False
5,0,hook_resid_mid,blocks.0.hook_resid_mid,blocks__0__hook_resid_mid,2833,29.190092,30.863335,7.339045,10.0,1.0,True,944,1.516403,0.151463,-1.054814,0.886761,1.364941,1.941575,1.653258,0.572929,0.532839,0.888771,10.0,2.770855,False,False
6,0,hook_resid_mid,blocks.0.hook_resid_mid,blocks__0__hook_resid_mid,2833,29.190092,30.863335,7.339045,7.5,1.0,True,944,1.516403,0.798030,-1.054814,0.279931,0.718374,1.334745,1.026559,0.440665,0.215042,0.690678,7.5,2.770855,False,False
7,0,hook_resid_pre,blocks.0.hook_resid_pre,blocks__0__hook_resid_pre,2833,4.719832,4.851473,1.466795,1.0,1.0,True,944,1.516403,1.170957,-1.054814,-0.303519,0.345447,0.751294,0.548371,0.264115,0.094280,0.346398,1.0,2.770855,False,False
8,1,hook_resid_post,blocks.1.hook_resid_post,blocks__1__hook_resid_post,2833,54.701641,56.779804,18.831623,10.0,1.0,True,944,1.516403,1.159219,-1.054814,-0.400574,0.357184,0.654240,0.505712,0.256397,0.091102,0.301907,10.0,2.770855,False,False
9,2,hook_resid_pre,blocks.2.hook_resid_pre,blocks__2__hook_resid_pre,2833,54.701641,56.779804,18.831623,10.0,1.0,True,944,1.516403,1.159219,-1.054814,-0.400574,0.357184,0.654240,0.505712,0.256397,0.091102,0.301907,10.0,2.770855,False,False


In [10]:
if not validation.empty:
    best_idx = validation.groupby(["layer", "hook_point"])["signed_effect_mean"].idxmax()
    best_by_site = validation.loc[best_idx].copy()
    fig = px.imshow(
        best_by_site.pivot(index="hook_point", columns="layer", values="signed_effect_mean"),
        color_continuous_scale="RdBu_r",
        aspect="auto",
        title="Best Validation Steering Effect by Layer and Residual Point",
        labels={"color": "best signed effect"},
    )
    fig.update_layout(height=360)
    fig.show()

In [11]:
if not validation.empty:
    filtered_validation = validation[validation["hook_name"] != "blocks.0.hook_resid_pre"]
    best_idx = filtered_validation.groupby(["layer", "hook_point"])["signed_effect_mean"].idxmax()
    best_by_site = filtered_validation.loc[best_idx].copy()
    fig = px.imshow(
        best_by_site.pivot(index="hook_point", columns="layer", values="signed_effect_mean"),
        color_continuous_scale="RdBu_r",
        aspect="auto",
        title="Best Validation Steering Effect (excluding blocks.0.hook_resid_pre)",
        labels={"color": "best signed effect"},
    )
    fig.update_layout(height=360)
    fig.show()

In [12]:
if not validation.empty:
    filtered_validation = validation[validation["layer"] != 0]
    best_idx = filtered_validation.groupby(["layer", "hook_point"])["signed_effect_mean"].idxmax()
    best_by_site = filtered_validation.loc[best_idx].copy()
    fig = px.imshow(
        best_by_site.pivot(index="hook_point", columns="layer", values="signed_effect_mean"),
        color_continuous_scale="RdBu_r",
        aspect="auto",
        title="Best Validation Steering Effect (excluding layer 0)",
        labels={"color": "best signed effect"},
    )
    fig.update_layout(height=360)
    fig.show()

In [13]:
if not validation.empty:
    best_idx = validation.groupby(["layer", "hook_point"])["signed_effect_mean"].idxmax()
    best_by_site = validation.loc[best_idx].copy()
    fig = px.imshow(
        best_by_site.pivot(index="hook_point", columns="layer", values="alpha"),
        color_continuous_scale="PRGn",
        aspect="auto",
        title="Alpha at Best Validation Effect by Layer and Residual Point",
        labels={"color": "alpha"},
    )
    fig.update_layout(height=360)
    fig.show()

In [14]:
if not validation.empty:
    selected = summary.get("selected") or selected_payload.get("selected") or {}
    top_layers = (
        validation.groupby("layer")["signed_effect_mean"]
        .max()
        .sort_values(ascending=False)
        .head(6)
        .index.tolist()
    )
    curves = validation[validation["layer"].isin(top_layers)].copy()
    curves["site"] = curves["hook_point"] + " L" + curves["layer"].astype(str)
    fig = px.line(
        curves.sort_values("alpha"),
        x="alpha",
        y="signed_effect_mean",
        color="site",
        markers=True,
        title="Validation Effect Across Alpha for Top Layers",
    )
    if selected:
        fig.add_trace(
            go.Scatter(
                x=[selected.get("alpha")],
                y=[selected.get("signed_effect_mean")],
                mode="markers",
                marker=dict(size=14, symbol="x", color="black"),
                name="selected",
            )
        )
    fig.show()

## Concept Vector Norms

In [15]:
if site_summary.empty:
    display(Markdown("Site summary is not available yet."))
else:
    fig = px.line(
        site_summary,
        x="layer",
        y="concept_norm",
        color="hook_point",
        markers=True,
        title="Raw Concept Vector Norm by Layer and Residual Point",
    )
    fig.show()

## Held-Out Test Effects

In [16]:
if test_rows.empty:
    display(Markdown("Test steering rows are not available yet. The run may still be in progress."))
else:
    display(test_rows.head())

,split,uid,clean_prefix,corrupt_prefix,patient,clean_verb,corrupt_verb,domain,hook_name,alpha,clean_metric_before,clean_metric_after,corrupt_metric_before,corrupt_metric_after,clean_signed_effect,corrupt_signed_effect,example_signed_effect,clean_flipped_below_zero,corrupt_flipped_above_zero
0,test,be5a1cb39ba2476b88d61b4fc314b856,The tram was parked by the,The tram was crushed by the,tram,parked,crushed,Vehicles and Conveyances,blocks.0.hook_resid_pre,5.0,1.100179,-1.539966,-1.122782,1.517861,2.640146,2.640643,2.640394,True,True
1,test,ea2ed51abbd2453fbce85019ef741ae1,The victim was examined by the,The victim was blinded by the,victim,examined,blinded,Humans as Physical Entities,blocks.0.hook_resid_pre,5.0,2.246454,-2.308762,-1.695235,2.572695,4.555216,4.267930,4.411573,True,True
2,test,ecf43cf1976b4a5eaa98309d2d7d19a3,The boy was assisted by the,The boy was crushed by the,boy,assisted,crushed,Humans as Physical Entities,blocks.0.hook_resid_pre,5.0,2.396213,-2.083697,-1.360358,2.551613,4.479910,3.911971,4.195940,True,True
3,test,6164e3c8e01e4db58a96ffcef6c267dd,The van was inspected by the,The van was shaken by the,van,inspected,shaken,Vehicles and Conveyances,blocks.0.hook_resid_pre,5.0,0.908238,-2.110221,-0.875358,1.832940,3.018459,2.708298,2.863379,True,True
4,test,ae3584a5a3f9440a93da85b2574ed736,The road was maintained by the,The road was battered by the,road,maintained,battered,Physical Structures,blocks.0.hook_resid_pre,5.0,0.618938,-1.924594,-1.234223,1.482406,2.543531,2.716630,2.630081,True,True


In [17]:
if not test_rows.empty:
    metric_long = []
    for condition in ["clean", "corrupt"]:
        for moment in ["before", "after"]:
            metric_long.append(
                pd.DataFrame(
                    {
                        "condition": condition,
                        "moment": moment,
                        "metric": test_rows[f"{condition}_metric_{moment}"],
                    }
                )
            )
    metric_long = pd.concat(metric_long, ignore_index=True)
    fig = px.box(
        metric_long,
        x="condition",
        y="metric",
        color="moment",
        points="all",
        title="Held-Out Metric Before vs After Steering",
    )
    fig.add_hline(y=0, line_dash="dash", line_color="black")
    fig.show()

In [18]:
if not test_rows.empty:
    effect_long = test_rows.melt(
        value_vars=["clean_signed_effect", "corrupt_signed_effect", "example_signed_effect"],
        var_name="effect_type",
        value_name="signed_effect",
    )
    fig = px.histogram(
        effect_long,
        x="signed_effect",
        color="effect_type",
        barmode="overlay",
        opacity=0.65,
        nbins=50,
        title="Held-Out Steering Effect Distribution",
    )
    fig.add_vline(x=0, line_dash="dash", line_color="black")
    fig.show()

In [19]:
if not test_rows.empty and "domain" in test_rows:
    by_domain = (
        test_rows.groupby("domain", dropna=False)
        .agg(
            examples=("example_signed_effect", "size"),
            mean_effect=("example_signed_effect", "mean"),
            clean_flip_rate=("clean_flipped_below_zero", "mean"),
            corrupt_flip_rate=("corrupt_flipped_above_zero", "mean"),
        )
        .reset_index()
        .sort_values("mean_effect", ascending=False)
    )
    display(by_domain)
    fig = px.bar(
        by_domain,
        x="domain",
        y="mean_effect",
        color="examples",
        title="Mean Held-Out Effect by Semantic Domain",
    )
    fig.update_layout(xaxis_tickangle=-35)
    fig.show()

,domain,examples,mean_effect,clean_flip_rate,corrupt_flip_rate
4,Humans as Physical Entities,213,3.775227,1.000000,1.0
5,Information Resources,28,3.279218,1.000000,1.0
9,Vehicles and Conveyances,182,3.117204,1.000000,1.0
8,Textile Garment Frame,39,2.940078,1.000000,1.0
1,Biological Organisms,48,2.896519,1.000000,1.0
7,Physical Structures,252,2.797376,1.000000,1.0
2,Culinary Frame,31,2.791750,1.000000,1.0
0,Academic Scientific Frame,87,2.665724,0.862069,1.0
6,Legal Administrative Frame,25,2.458931,0.520000,1.0
3,Financial Economic Frame,39,2.290565,1.000000,1.0


## Random Direction Control

In [20]:
if random_control.empty:
    display(Markdown("Random control summary is not available yet."))
else:
    display(random_control)
    selected_effect = summary.get("test_summary", {}).get("signed_effect_mean")
    fig = px.histogram(
        random_control,
        x="signed_effect_mean",
        nbins=max(5, min(20, len(random_control))),
        title="Random Direction Control vs Selected Concept Direction",
    )
    if selected_effect is not None:
        fig.add_vline(
            x=selected_effect,
            line_color="red",
            line_width=3,
            annotation_text="selected concept",
        )
    fig.show()

,repeat,hook_name,alpha,control_vector_norm,example_count,clean_before_mean,clean_after_mean,corrupt_before_mean,corrupt_after_mean,clean_signed_effect_mean,corrupt_signed_effect_mean,signed_effect_mean,signed_effect_std,clean_flip_rate,corrupt_flip_rate
0,0,blocks.0.hook_resid_pre,5.0,1.0,944,1.516664,1.571319,-1.041097,-1.129948,-0.054656,-0.088851,-0.071753,0.222899,0.000000,0.026483
1,1,blocks.0.hook_resid_pre,5.0,1.0,944,1.516664,1.666595,-1.041097,-1.111989,-0.149931,-0.070891,-0.110411,0.251301,0.000000,0.027542
2,2,blocks.0.hook_resid_pre,5.0,1.0,944,1.516664,1.657073,-1.041097,-1.400484,-0.140409,-0.359387,-0.249898,0.219242,0.000000,0.009534
3,3,blocks.0.hook_resid_pre,5.0,1.0,944,1.516664,2.043429,-1.041097,-1.154151,-0.526765,-0.113053,-0.319909,0.291711,0.000000,0.023305
4,4,blocks.0.hook_resid_pre,5.0,1.0,944,1.516664,1.517674,-1.041097,-0.819638,-0.001010,0.221460,0.110225,0.270468,0.002119,0.117585
5,5,blocks.0.hook_resid_pre,5.0,1.0,944,1.516664,1.146077,-1.041097,-0.510759,0.370587,0.530338,0.450463,0.315037,0.132415,0.208686
6,6,blocks.0.hook_resid_pre,5.0,1.0,944,1.516664,1.415702,-1.041097,-1.077420,0.100962,-0.036323,0.032319,0.223134,0.000000,0.033898
7,7,blocks.0.hook_resid_pre,5.0,1.0,944,1.516664,1.136273,-1.041097,-0.309194,0.380391,0.731903,0.556147,0.314239,0.099576,0.277542
8,8,blocks.0.hook_resid_pre,5.0,1.0,944,1.516664,1.784697,-1.041097,-0.439075,-0.268033,0.602022,0.166994,0.267622,0.000000,0.272246
9,9,blocks.0.hook_resid_pre,5.0,1.0,944,1.516664,1.604260,-1.041097,-0.888878,-0.087596,0.152219,0.032312,0.290723,0.000000,0.131356


## Strongest Examples

In [21]:
if not test_rows.empty:
    columns = [
        "example_signed_effect",
        "clean_signed_effect",
        "corrupt_signed_effect",
        "clean_prefix",
        "corrupt_prefix",
        "patient",
        "clean_verb",
        "corrupt_verb",
        "domain",
    ]
    display(Markdown("### Largest positive effects"))
    display(test_rows.sort_values("example_signed_effect", ascending=False)[columns].head(20))
    display(Markdown("### Largest negative effects"))
    display(test_rows.sort_values("example_signed_effect", ascending=True)[columns].head(20))

### Largest positive effects

,example_signed_effect,clean_signed_effect,corrupt_signed_effect,clean_prefix,corrupt_prefix,patient,clean_verb,corrupt_verb,domain
469,4.695454,4.854042,4.536865,The girl was identified by the,The girl was struck by the,girl,identified,struck,Humans as Physical Entities
154,4.692853,4.885684,4.500022,The survivor was interviewed by the,The survivor was struck by the,survivor,interviewed,struck,Humans as Physical Entities
293,4.674763,4.992831,4.356696,The girl was interviewed by the,The girl was blinded by the,girl,interviewed,blinded,Humans as Physical Entities
395,4.605369,4.854043,4.356696,The girl was identified by the,The girl was blinded by the,girl,identified,blinded,Humans as Physical Entities
18,4.597805,4.658745,4.536865,The girl was examined by the,The girl was struck by the,girl,examined,struck,Humans as Physical Entities
145,4.568278,4.852463,4.284093,The boy was identified by the,The boy was frozen by the,boy,identified,frozen,Humans as Physical Entities
429,4.553819,4.736765,4.370873,The boy was examined by the,The boy was blinded by the,boy,examined,blinded,Humans as Physical Entities
735,4.546460,4.793233,4.299686,The tourist was interviewed by the,The tourist was blinded by the,tourist,interviewed,blinded,Humans as Physical Entities
756,4.529184,4.917053,4.141315,The farmer was interviewed by the,The farmer was crushed by the,farmer,interviewed,crushed,Humans as Physical Entities
36,4.496382,4.917053,4.075711,The farmer was interviewed by the,The farmer was blinded by the,farmer,interviewed,blinded,Humans as Physical Entities


### Largest negative effects

,example_signed_effect,clean_signed_effect,corrupt_signed_effect,clean_prefix,corrupt_prefix,patient,clean_verb,corrupt_verb,domain
284,1.121203,0.525415,1.716992,The protocol was validated by the,The protocol was degraded by the,protocol,validated,degraded,Academic Scientific Frame
115,1.359437,0.723997,1.994877,The dataset was validated by the,The dataset was degraded by the,dataset,validated,degraded,Academic Scientific Frame
924,1.427622,0.115425,2.739819,The thesis was validated by the,The thesis was damaged by the,thesis,validated,damaged,Academic Scientific Frame
32,1.476213,1.165955,1.786471,The revenue was protected by the,The revenue was depleted by the,revenue,protected,depleted,Financial Economic Frame
555,1.476948,0.642023,2.311874,The report was validated by the,The report was obscured by the,report,validated,obscured,Academic Scientific Frame
180,1.490901,0.052741,2.929062,The abstract was validated by the,The abstract was obscured by the,abstract,validated,obscured,Academic Scientific Frame
322,1.510109,1.250395,1.769824,The savings was protected by the,The savings was depleted by the,savings,protected,depleted,Financial Economic Frame
75,1.554654,0.942607,2.166702,The graph was validated by the,The graph was degraded by the,graph,validated,degraded,Academic Scientific Frame
316,1.584747,1.642567,1.526928,The surplus was adjusted by the,The surplus was depleted by the,surplus,adjusted,depleted,Financial Economic Frame
29,1.608334,0.011019,3.205650,The paper was validated by the,The paper was damaged by the,paper,validated,damaged,Academic Scientific Frame


## Activation Geometry At The Selected Site

This section is post-hoc. It reloads only the held-out test split, re-runs the model at the selected hook, and extracts only the verb-token activations at that single site. The projection onto the selected concept direction is the primary evidence; the 2D PCA view is only secondary intuition.

In [22]:
import sys

NOTEBOOK_DIR = PROJECT_ROOT / "scripts" / "notebooks"
EXECUTABLE_DIR = PROJECT_ROOT / "scripts" / "executable"
for path in (EXECUTABLE_DIR, NOTEBOOK_DIR):
    path_str = str(path)
    if path_str not in sys.path:
        sys.path.insert(0, path_str)

from circuit_finder_core import concept_steering_vector

import torch


In [23]:
selected_info = summary.get("selected") or selected_payload.get("selected") or {}
selected_hook = selected_info.get("hook_name")
selected_alpha = selected_info.get("alpha")
selected_hook

'blocks.0.hook_resid_pre'

In [24]:
if not selected_hook:
    raise ValueError("No selected hook found in the current run artifacts.")

import torch

vector_path = latest_file("concept_vectors_*.pt", required=True)
vector_payload = torch.load(vector_path, map_location="cpu")
raw_vector = vector_payload["vectors"][selected_hook].float().cpu()
normalize_flag = bool((summary.get("config") or {}).get("normalize_concept_vector", True))
steering_vector = concept_steering_vector(raw_vector, normalize=normalize_flag)
print({
    "selected_hook": selected_hook,
    "selected_alpha": selected_alpha,
    "raw_norm": float(raw_vector.norm().item()),
    "steering_norm": float(steering_vector.norm().item()),
    "normalize_concept_vector": normalize_flag,
})

{'selected_hook': 'blocks.0.hook_resid_pre', 'selected_alpha': 5.0, 'raw_norm': 1.4667949676513672, 'steering_norm': 0.9999999403953552, 'normalize_concept_vector': True}


In [25]:
ACTIVATION_BATCH_SIZE = 64
MODEL_NAME_FOR_EXTRACTION = summary.get("model_name", MODEL_SLUG)

activation_geometry_ready = not projection_df.empty
activation_geometry_error = projection_summary_payload.get("error")
model = None
clean_acts = None
corrupt_acts = None
pair_diffs = None
records = projection_df.to_dict("records") if not projection_df.empty else []

display(pd.DataFrame([
    {
        "analysis_manifest": str(paths.get("analysis_manifest")) if paths.get("analysis_manifest") else None,
        "projection_rows": int(len(projection_df)),
        "pca_points": int(len(pca_points)),
        "selected_hook": selected_hook,
        "selected_alpha": selected_alpha,
        "device": projection_summary_payload.get("device"),
        "test_examples": projection_summary_payload.get("test_examples"),
    }
]).T.rename(columns={0: "value"}))


,value
analysis_manifest,/gpfs/home4/spunzo/grammatical-circuits/animacy-circuit/results/concept_extraction/gpt2/2026-05-30/concept_analysis_artifacts_2026-06-06_203315.json
projection_rows,944
pca_points,1888
selected_hook,blocks.0.hook_resid_pre
selected_alpha,5.0
device,cuda
test_examples,944


In [26]:
if not activation_geometry_ready:
    display(Markdown(
        "Activation-geometry artifacts are not available yet. Run "
        f"`python animacy-circuit/scripts/executable/run_concept_analysis_artifacts.py --model {MODEL_SLUG} --run-dir {RUN_DIR}` "
        "and then reload this notebook."
    ))
else:
    display(Markdown("Loaded saved activation-geometry artifacts."))


Loaded saved activation-geometry artifacts.

In [27]:
if not activation_geometry_ready:
    print("Skipping projection table because saved activation-geometry artifacts are not available.")
else:
    display(projection_df[[
        "clean_prefix", "corrupt_prefix", "patient", "clean_verb", "corrupt_verb", "domain",
        "clean_proj", "corrupt_proj", "pair_diff_proj"
    ]].head())


,clean_prefix,corrupt_prefix,patient,clean_verb,corrupt_verb,domain,clean_proj,corrupt_proj,pair_diff_proj
0,The tram was parked by the,The tram was crushed by the,tram,parked,crushed,Vehicles and Conveyances,-0.199154,-1.268703,1.069550
1,The victim was examined by the,The victim was blinded by the,victim,examined,blinded,Humans as Physical Entities,0.685346,-1.089728,1.775074
2,The boy was assisted by the,The boy was crushed by the,boy,assisted,crushed,Humans as Physical Entities,0.375079,-1.268703,1.643783
3,The van was inspected by the,The van was shaken by the,van,inspected,shaken,Vehicles and Conveyances,0.848080,-1.130917,1.978997
4,The road was maintained by the,The road was battered by the,road,maintained,battered,Physical Structures,0.406100,-1.215742,1.621842


In [28]:
if not activation_geometry_ready:
    print("Skipping projection histogram because activation extraction did not run.")
else:
    proj_long = pd.concat(
        [
            pd.DataFrame({"sentence_type": "clean", "projection": projection_df["clean_proj"]}),
            pd.DataFrame({"sentence_type": "corrupt", "projection": projection_df["corrupt_proj"]}),
        ],
        ignore_index=True,
    )
    fig = px.histogram(
        proj_long,
        x="projection",
        color="sentence_type",
        barmode="overlay",
        opacity=0.65,
        nbins=50,
        title="Projection Onto The Selected Concept Direction",
    )
    fig.add_vline(x=0, line_dash="dash", line_color="black")
    fig.show()


In [29]:
if not activation_geometry_ready:
    print("Skipping pair-difference histogram because activation extraction did not run.")
else:
    fig = px.histogram(
        projection_df,
        x="pair_diff_proj",
        nbins=50,
        title="Held-Out Pair Difference Along The Selected Concept Direction: (clean - corrupt) · v",
    )
    fig.add_vline(x=0, line_dash="dash", line_color="black")
    fig.show()


In [30]:
if not activation_geometry_ready:
    print("Skipping paired projection plot because activation extraction did not run.")
else:
    paired_plot = projection_df.reset_index().rename(columns={"index": "pair_id"})
    fig = go.Figure()
    for _, row in paired_plot.iterrows():
        fig.add_trace(
            go.Scatter(
                x=["clean", "corrupt"],
                y=[row["clean_proj"], row["corrupt_proj"]],
                mode="lines+markers",
                line=dict(color="rgba(80,80,80,0.35)"),
                marker=dict(size=6),
                showlegend=False,
                customdata=[[row.get("patient"), row.get("clean_verb"), row.get("corrupt_verb")]] * 2,
                hovertemplate="patient=%{customdata[0]}<br>clean_verb=%{customdata[1]}<br>corrupt_verb=%{customdata[2]}<br>projection=%{y:.3f}<extra></extra>",
            )
        )
    fig.update_layout(title="Per-Pair Projection Shift At The Selected Site", yaxis_title="projection onto concept direction")
    fig.show()


In [31]:
if pca_points.empty:
    print("Skipping PCA scatter because saved PCA points are not available.")
else:
    fig = px.scatter(
        pca_points,
        x="pc1",
        y="pc2",
        color="sentence_type",
        hover_name="active_verb",
        hover_data={
            "pair_id": True,
            "sentence_type": True,
            "active_verb": True,
            "active_sentence": True,
            "patient": True,
            "domain": True,
            "clean_verb": False,
            "corrupt_verb": False,
            "clean_prefix": False,
            "corrupt_prefix": False,
            "pc1": ":.3f",
            "pc2": ":.3f",
        },
        title="2D PCA Of Verb Activations At The Selected Site",
    )
    fig.show()


In [32]:
if not activation_geometry_ready:
    print("Skipping projection summary because saved activation-geometry artifacts are not available.")
else:
    display(pd.DataFrame([projection_summary_payload]).T.rename(columns={0: "value"}))


,value
clean_proj_mean,0.318482
corrupt_proj_mean,-1.156798
pair_diff_proj_mean,1.47528
pair_diff_proj_std,0.430558
test_steering_signed_effect_mean,3.062685
random_control_signed_effect_mean,0.059649
example_count,944
selected_hook,blocks.0.hook_resid_pre
pca_status,"{'available': True, 'error': None}"
device,cuda


## BLiMP Transfer Evaluation

This section is also visualization-only now. Precompute the BLiMP transfer artifacts with `animacy-circuit/scripts/executable/run_concept_analysis_artifacts.py`, then reload this notebook to inspect the saved summaries, plots, and example tables.


In [33]:
BLIMP_CONFIG = blimp_status.get("config", "animate_subject_trans") if blimp_status else "animate_subject_trans"
BLIMP_BATCH_SIZE = None
BLIMP_RANDOM_CONTROL_REPEATS = blimp_status.get("random_control_repeats") if blimp_status else None
BLIMP_DISPLAY_CONDITIONS = ["main_verb", "main_verb_bad_only"]
blimp_optional_artifacts = {
    "status_path": str(paths.get("blimp_transfer_status")) if paths.get("blimp_transfer_status") else None,
    "rows_path": str(paths.get("blimp_transfer_rows")) if paths.get("blimp_transfer_rows") else None,
    "summary_path": str(paths.get("blimp_transfer_summary")) if paths.get("blimp_transfer_summary") else None,
    "random_summary_path": str(paths.get("blimp_transfer_random_summary")) if paths.get("blimp_transfer_random_summary") else None,
}
blimp_ready = bool(blimp_status.get("ready")) and not blimp_results.empty
blimp_ready_reason = blimp_status.get("reason") if blimp_status else None


In [34]:
display(pd.DataFrame([blimp_status]).T.rename(columns={0: "value"}))
if not blimp_alignment_failures.empty:
    display(Markdown("First BLiMP alignment failures:"))
    display(blimp_alignment_failures.head(10))
if not blimp_ready:
    display(Markdown(
        "BLiMP transfer artifacts are not available yet. Run "
        f"`python animacy-circuit/scripts/executable/run_concept_analysis_artifacts.py --model {MODEL_SLUG} --run-dir {RUN_DIR}` "
        "and then reload this notebook."
    ))


,value
config,animate_subject_passive
ready,True
reason,None
raw_rows,1000
aligned_rows,967
alignment_failures,33
validation_rows,100
evaluation_rows,867
position_policy,main_verb_single_token_only
selected_hook,blocks.0.hook_resid_pre


First BLiMP alignment failures:

,row,UID,sentence_good,sentence_bad,alignment_error,good_subject_error,bad_subject_error,good_verb_error,bad_verb_error
0,86,animate_subject_passive,These universities were boycotted by some lady.,These universities were boycotted by some mirrors.,model_token_alignment_failed,NaN,NaN,token_not_single_model_token,token_not_single_model_token
1,118,animate_subject_passive,This mountain wasn't biked to by some children.,This mountain wasn't biked to by some drawings.,model_token_alignment_failed,NaN,NaN,token_not_single_model_token,token_not_single_model_token
2,158,animate_subject_passive,Hospitals aren't biked to by the man.,Hospitals aren't biked to by the mouth.,model_token_alignment_failed,NaN,NaN,token_not_single_model_token,token_not_single_model_token
3,258,animate_subject_passive,Every mountain is biked to by the women.,Every mountain is biked to by the gloves.,model_token_alignment_failed,NaN,NaN,token_not_single_model_token,token_not_single_model_token
4,296,animate_subject_passive,Those schools aren't biked to by some senators.,Those schools aren't biked to by some box.,model_token_alignment_failed,NaN,NaN,token_not_single_model_token,token_not_single_model_token
5,335,animate_subject_passive,A lot of organizations were boycotted by the adult.,A lot of organizations were boycotted by the university.,model_token_alignment_failed,NaN,NaN,token_not_single_model_token,token_not_single_model_token
6,361,animate_subject_passive,The mountain was boycotted by the man.,The mountain was boycotted by the nuclei.,model_token_alignment_failed,NaN,token_not_single_model_token,token_not_single_model_token,token_not_single_model_token
7,367,animate_subject_passive,A grocery store isn't skated around by some teenager.,A grocery store isn't skated around by some light.,model_token_alignment_failed,NaN,NaN,token_not_single_model_token,token_not_single_model_token
8,383,animate_subject_passive,A lot of rivers are boycotted by some waitress.,A lot of rivers are boycotted by some university.,model_token_alignment_failed,NaN,NaN,token_not_single_model_token,token_not_single_model_token
9,417,animate_subject_passive,Most museums were biked to by the women.,Most museums were biked to by the scarf.,model_token_alignment_failed,NaN,NaN,token_not_single_model_token,token_not_single_model_token


In [35]:
if blimp_summary.empty:
    print("Skipping BLiMP summary because saved transfer artifacts are not available.")
else:
    primary_controls = blimp_summary[
        blimp_summary["control_label"].isin(["selected_direction", "shuffled_label_direction"])
        & blimp_summary["condition"].isin(BLIMP_DISPLAY_CONDITIONS)
    ].copy()
    if ("subject_head" in BLIMP_DISPLAY_CONDITIONS) and not blimp_random_summary.empty:
        random_aggregate = {
            "condition": "subject_head",
            "control_label": "random_direction_mean",
            "repeat": None,
            "example_count": int(blimp_random_summary["example_count"].iloc[0]),
            "baseline_score_mean": float(blimp_random_summary["baseline_score_mean"].mean()),
            "steered_score_mean": float(blimp_random_summary["steered_score_mean"].mean()),
            "score_shift_mean": float(blimp_random_summary["score_shift_mean"].mean()),
            "score_shift_std": float(blimp_random_summary["score_shift_mean"].std(ddof=0)) if len(blimp_random_summary) > 1 else 0.0,
            "baseline_accuracy": float(blimp_random_summary["baseline_accuracy"].mean()),
            "steered_accuracy": float(blimp_random_summary["steered_accuracy"].mean()),
            "accuracy_shift": float(blimp_random_summary["accuracy_shift"].mean()),
            "baseline_correct_count": int(blimp_random_summary["baseline_correct_count"].iloc[0]),
            "baseline_wrong_count": int(blimp_random_summary["baseline_wrong_count"].iloc[0]),
            "flip_to_good_count": float(blimp_random_summary["flip_to_good_count"].mean()),
            "flip_away_from_good_count": float(blimp_random_summary["flip_away_from_good_count"].mean()),
            "flip_to_good_rate": float(blimp_random_summary["flip_to_good_rate"].mean()),
            "flip_away_from_good_rate": float(blimp_random_summary["flip_away_from_good_rate"].mean()),
            "flip_to_good_given_baseline_wrong": float(blimp_random_summary["flip_to_good_given_baseline_wrong"].mean()),
            "flip_away_given_baseline_correct": float(blimp_random_summary["flip_away_given_baseline_correct"].mean()),
        }
        primary_controls = pd.concat([primary_controls, pd.DataFrame([random_aggregate])], ignore_index=True)

    display(Markdown("### BLiMP condition summary"))
    display(primary_controls[[
        "condition",
        "control_label",
        "example_count",
        "baseline_accuracy",
        "steered_accuracy",
        "accuracy_shift",
        "baseline_score_mean",
        "steered_score_mean",
        "score_shift_mean",
        "score_shift_std",
        "baseline_correct_count",
        "baseline_wrong_count",
        "flip_to_good_count",
        "flip_away_from_good_count",
        "flip_to_good_rate",
        "flip_away_from_good_rate",
        "flip_to_good_given_baseline_wrong",
        "flip_away_given_baseline_correct",
    ]].sort_values(["condition", "control_label"]).reset_index(drop=True))

    if ("subject_head" in BLIMP_DISPLAY_CONDITIONS) and not blimp_random_summary.empty:
        display(Markdown("### Random-direction subject-head repeats"))
        display(blimp_random_summary[[
            "repeat",
            "baseline_accuracy",
            "steered_accuracy",
            "accuracy_shift",
            "score_shift_mean",
            "score_shift_std",
        ]])


### BLiMP condition summary

,condition,control_label,example_count,baseline_accuracy,steered_accuracy,accuracy_shift,baseline_score_mean,steered_score_mean,score_shift_mean,score_shift_std,baseline_correct_count,baseline_wrong_count,flip_to_good_count,flip_away_from_good_count,flip_to_good_rate,flip_away_from_good_rate,flip_to_good_given_baseline_wrong,flip_away_given_baseline_correct
0,main_verb,selected_direction,867,0.795848,0.838524,0.042676,3.202463,3.598022,0.395559,1.420786,690,177,48,11,0.055363,0.012687,0.271186,0.015942
1,main_verb_bad_only,selected_direction,867,0.795848,0.786621,-0.009227,3.202463,3.274534,0.072071,1.930970,690,177,37,45,0.042676,0.051903,0.209040,0.065217


In [36]:
if blimp_summary.empty:
    print("Skipping BLiMP plots because no evaluation rows are available.")
else:
    summary_plot_df = blimp_summary[
        blimp_summary["control_label"].isin(["selected_direction", "shuffled_label_direction"])
        & blimp_summary["condition"].isin(BLIMP_DISPLAY_CONDITIONS)
    ].copy()
    if not blimp_random_summary.empty:
        summary_plot_df = pd.concat([
            summary_plot_df,
            pd.DataFrame([{
                "condition": "subject_head",
                "control_label": "random_direction_mean",
                "score_shift_mean": float(blimp_random_summary["score_shift_mean"].mean()),
                "accuracy_shift": float(blimp_random_summary["accuracy_shift"].mean()),
            }]),
        ], ignore_index=True)

    fig = px.bar(
        summary_plot_df.sort_values(["condition", "control_label"]),
        x="condition",
        y="score_shift_mean",
        color="control_label",
        barmode="group",
        title="BLiMP Preference Shift: Mean(score_after - score_before)",
        hover_data={"accuracy_shift": ":.4f"},
    )
    fig.add_hline(y=0, line_dash="dash", line_color="black")
    fig.show()

    selected_rows = blimp_results[
        (blimp_results["control_label"] == "selected_direction")
        & blimp_results["condition"].isin(BLIMP_DISPLAY_CONDITIONS)
    ].copy()
    fig = px.box(
        selected_rows,
        x="condition",
        y="score_shift",
        points="outliers",
        title="BLiMP Per-Example Preference Shift For The Frozen Selected Direction",
    )
    fig.add_hline(y=0, line_dash="dash", line_color="black")
    fig.show()


In [37]:
if blimp_results.empty:
    print("Skipping BLiMP example tables because no evaluation rows are available.")
else:
    selected_subject_rows = blimp_results[
        (blimp_results["condition"] == "main_verb_bad_only")
        & (blimp_results["control_label"] == "selected_direction")
    ].copy()
    display(Markdown("### Largest positive main-verb bad-only preference shifts"))
    display(selected_subject_rows.sort_values("score_shift", ascending=False)[[
        "sentence_good",
        "sentence_bad",
        "subject_head_good",
        "subject_head_bad",
        "main_verb",
        "baseline_score",
        "steered_score",
        "score_shift",
        "baseline_prefers_good",
        "steered_prefers_good",
    ]].head(20))

    display(Markdown("### Largest negative main-verb bad-only preference shifts"))
    display(selected_subject_rows.sort_values("score_shift", ascending=True)[[
        "sentence_good",
        "sentence_bad",
        "subject_head_good",
        "subject_head_bad",
        "main_verb",
        "baseline_score",
        "steered_score",
        "score_shift",
        "baseline_prefers_good",
        "steered_prefers_good",
    ]].head(20))


### Largest positive main-verb bad-only preference shifts

,sentence_good,sentence_bad,subject_head_good,subject_head_bad,main_verb,baseline_score,steered_score,score_shift,baseline_prefers_good,steered_prefers_good
1542,Curtis is concealed by some teenager.,Curtis is concealed by some paralysis.,teenager,paralysis,concealed,-0.333446,7.274651,7.608097,False,True
1395,Elaine isn't escaped from by the pedestrians.,Elaine isn't escaped from by the mouth.,pedestrians,mouth,escaped,-2.674324,4.659637,7.333962,False,True
927,Martin wasn't escaped from by some offspring.,Martin wasn't escaped from by some mouse.,offspring,mouse,escaped,-0.737675,6.359524,7.097199,False,True
1282,Victoria is attacked by some person.,Victoria is attacked by some icicles.,person,icicles,attacked,1.318424,8.246212,6.927788,True,True
1455,These doctors weren't escaped from by some actress.,These doctors weren't escaped from by some lamps.,actress,lamps,escaped,2.734604,9.525139,6.790535,True,True
1049,James wasn't escaped from by the women.,James wasn't escaped from by the cart.,women,cart,escaped,2.077545,8.273582,6.196037,True,True
1154,Jennifer is healed by some waiter.,Jennifer is healed by some truck.,waiter,truck,healed,0.403080,6.427059,6.023979,True,True
1493,Most shoes weren't messed up by the guests.,Most shoes weren't messed up by the pictures.,guests,pictures,messed,-1.128590,4.881828,6.010418,False,True
1054,Every dish was messed up by the man.,Every dish was messed up by the spotlights.,man,spotlights,messed,5.935974,11.934364,5.998390,True,True
996,Jane was fled from by some actors.,Jane was fled from by some means.,actors,means,fled,-1.067657,4.867947,5.935604,False,True


### Largest negative main-verb bad-only preference shifts

,sentence_good,sentence_bad,subject_head_good,subject_head_bad,main_verb,baseline_score,steered_score,score_shift,baseline_prefers_good,steered_prefers_good
1059,A closet is gone to by some man.,A closet is gone to by some sketches.,man,sketches,gone,4.804672,-2.995068,-7.799740,True,False
1637,The cafes weren't gone to by some student.,The cafes weren't gone to by some horse.,student,horse,gone,3.222809,-4.304054,-7.526863,True,False
1458,All planes were broken by some dancers.,All planes were broken by some government.,dancers,government,broken,-1.342026,-8.010399,-6.668373,False,False
1529,Guests were thought about by some guests.,Guests were thought about by some picture.,guests,picture,thought,11.007805,4.626320,-6.381485,True,True
1144,A museum is exited by the waiters.,A museum is exited by the government.,waiters,government,exited,-2.591244,-7.976578,-5.385334,False,False
1292,Those lakes were gone to by the pedestrians.,Those lakes were gone to by the plate.,pedestrians,plate,gone,-0.656956,-5.959392,-5.302437,False,False
1545,Colleen wasn't sounded like by some boy.,Colleen wasn't sounded like by some dish.,boy,dish,like,4.655396,-0.081650,-4.737045,True,False
1715,All museums were returned to by some guest.,All museums were returned to by some shoe.,guest,shoe,returned,4.481693,-0.239178,-4.720871,True,False
1223,This pedestrian isn't thought about by the senators.,This pedestrian isn't thought about by the pamphlet.,senators,pamphlet,thought,3.173088,-1.473766,-4.646854,True,False
1440,Rivers weren't gone to by the teenagers.,Rivers weren't gone to by the print.,teenagers,print,gone,3.478283,-1.066387,-4.544670,True,False


In [38]:
if blimp_results.empty:
    print("Skipping BLiMP margin-binned flip analysis because no evaluation rows are available.")
else:
    margin_rows = blimp_results[
        (blimp_results["condition"] == "main_verb_bad_only")
        & (blimp_results["control_label"] == "selected_direction")
        & (blimp_results["baseline_prefers_good"].astype(bool))
    ].copy()
    if margin_rows.empty:
        print("No baseline-correct main_verb_bad_only rows are available for margin-binned analysis.")
    else:
        margin_rows["margin_bin"] = pd.qcut(
            margin_rows["baseline_score"],
            q=4,
            labels=["Q1", "Q2", "Q3", "Q4"],
            duplicates="drop",
        )
        margin_summary = margin_rows.groupby("margin_bin", observed=True).agg(
            n=("baseline_score", "size"),
            baseline_min=("baseline_score", "min"),
            baseline_max=("baseline_score", "max"),
            baseline_mean=("baseline_score", "mean"),
            steered_mean=("steered_score", "mean"),
            score_shift_mean=("score_shift", "mean"),
            flip_to_bad_n=("steered_prefers_good", lambda s: int((~s.astype(bool)).sum())),
            flip_to_bad_rate=("steered_prefers_good", lambda s: float((~s.astype(bool)).mean())),
        ).reset_index()
        display(Markdown("### Margin-binned main-verb bad-only flip analysis"))
        display(margin_summary)

        fig = px.bar(
            margin_summary,
            x="margin_bin",
            y="flip_to_bad_rate",
            hover_data={
                "n": True,
                "baseline_min": ":.3f",
                "baseline_max": ":.3f",
                "baseline_mean": ":.3f",
                "score_shift_mean": ":.3f",
            },
            title="Main-Verb Bad-Only Flip Rate By Baseline Margin Quartile",
        )
        fig.add_hline(y=0, line_dash="dash", line_color="black")
        fig.show()


### Margin-binned main-verb bad-only flip analysis

,margin_bin,n,baseline_min,baseline_max,baseline_mean,steered_mean,score_shift_mean,flip_to_bad_n,flip_to_bad_rate
0,Q1,173,0.027378,2.243504,1.177604,1.582540,0.404935,35,0.202312
1,Q2,172,2.268600,4.165108,3.252715,3.260733,0.008018,7,0.040698
2,Q3,172,4.173950,6.575333,5.213767,5.094196,-0.119571,3,0.017442
3,Q4,173,6.577774,17.070389,8.839183,8.449875,-0.389309,0,0.000000
